In [ ]:
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

In [41]:
data = pd.read_csv('../data/processed/train.csv')

y = data['LogSalePrice']
X = data.drop(columns=['LogSalePrice'])

In [ ]:

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:

models = {
    'LinearRegression': LinearRegression(),
    'Lasso': Lasso(random_state=42),
    'Ridge': Ridge(random_state=42),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42),
    'LGBM': LGBMRegressor(verbose=-1, random_state=42),
    'XGBoost': XGBRegressor(random_state=42),
    'CatBoost': CatBoostRegressor(verbose=0, random_state=42)
}

## Сравнение с параметрами по умолчанию

In [ ]:
results_without_params = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='neg_root_mean_squared_error'
    )

    # sklearn возвращает RMSE со знаком минус, поэтому меняем знак
    results_without_params.append(
        {'model_name': name,
         'rmse_mean': -scores.mean(),
         'rmse_std': scores.std()
         }
    )

results_without_params = pd.DataFrame(results_without_params)
results_without_params.sort_values('rmse_mean')

TypeError: got an unexpected keyword argument 'random_state'

## Подбор параметров

`RandomizedSearchCV` берёт несколько случайных комбинаций из сетки (по умолчанию 10), проверяет каждую на кросс-валидации и запоминает лучшую.

In [ ]:
param_grids = {
    'LinearRegression': {},

    'Lasso': {
        'alpha': [0.0001, 0.0003, 0.0005, 0.001, 0.003, 0.01],
        'max_iter': [50000],
    },

    'Ridge': {
        'alpha': [0.1, 1, 3, 10, 30, 100],
    },

    'DecisionTree': {
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_leaf': [1, 5, 10, 20],
    },

    'RandomForest': {
        'n_estimators': [300],
        'max_depth': [None, 20],
        'max_features': [0.2, 0.33, 0.5, 'sqrt'],
        'min_samples_leaf': [1, 2, 4],
    },

    'LGBM': {
        'n_estimators': [500, 1500],
        'learning_rate': [0.01, 0.05],
        'num_leaves': [7, 15, 31],
        'min_child_samples': [10, 20],
        'colsample_bytree': [0.3, 0.6],
    },

    'XGBoost': {
        'n_estimators': [500, 1500],
        'learning_rate': [0.01, 0.05],
        'max_depth': [3, 4, 6],
        'subsample': [0.7, 1.0],
        'colsample_bytree': [0.3, 0.6],
    },

    'CatBoost': {
        'iterations': [1000],
        'learning_rate': [0.03, 0.06],
        'depth': [4, 6, 8],
        'l2_leaf_reg': [1, 3, 10],
    },
}

In [ ]:
results = []
best_models = {}

for name, model in models.items():
    rgs = RandomizedSearchCV(
        model,
        param_grids[name],
        cv=cv,
        scoring='neg_root_mean_squared_error',
        random_state=42
    )
    rgs.fit(X, y)

    results.append(
        {'model_name': name,
         'rmse': -rgs.best_score_,
         'best_params': rgs.best_params_}
    )

results = pd.DataFrame(results)
results.sort_values('rmse')

c:\Users\User\miniconda3\envs\houseprices\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\User\miniconda3\envs\houseprices\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\User\miniconda3\envs\houseprices\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 6 is smaller than n_iter=10. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,model_name,rmse,best_params
6,XGBoost,0.131780,"{'subsample': 0.7, 'n_estimators': 500, 'max_d..."
7,CatBoost,0.132177,"{'learning_rate': 0.03, 'l2_leaf_reg': 10, 'it..."
1,Lasso,0.134567,"{'max_iter': 50000, 'alpha': 0.0003}"
5,LGBM,0.135461,"{'num_leaves': 15, 'n_estimators': 1500, 'min_..."
2,Ridge,0.136371,{'alpha': 3}
0,LinearRegression,0.139869,{}
4,RandomForest,0.142762,"{'n_estimators': 300, 'min_samples_leaf': 2, '..."
3,DecisionTree,0.185323,"{'min_samples_leaf': 10, 'max_depth': None}"
